## 2. Build Model

# DETR3D Architecture Overview

                             ┌────────────────────────────┐
                             │   Multi-View Images (RGB)  │
                             └────────────┬───────────────┘
                                          │
                        ┌─────────────────▼─────────────────┐
                        │     CNN Backbone (e.g., ResNet)   │
                        └─────────────────┬─────────────────┘
                                          │
                        ┌─────────────────▼─────────────────┐
                        │        FPN / Image Neck           │
                        │ (multi-scale feature extraction)  │
                        └─────────────────┬─────────────────┘
                                          │
                    ┌─────────────────────▼─────────────────────┐
                    │   2D Feature Maps from Multiple Cameras   │
                    └─────────────────────┬─────────────────────┘
                                          │
                                          ▼
                          ┌─────────────────────────────┐
                          │     Learned Object Queries   │
                          │     (with 3D reference pts)  │
                          └─────────────┬───────────────┘
                                        │
         ┌──────────────────────────────▼─────────────────────────────┐
         │ Project 3D reference points into 2D coords in each camera  │
         └──────────────────────────────┬─────────────────────────────┘
                                        │
        ┌───────────────────────────────▼──────────────────────────────┐
        │ Sample 2D features at projected coords from each camera view │
        └───────────────────────────────┬──────────────────────────────┘
                                        │
                       ┌────────────────▼────────────────┐
                       │  Fuse sampled features (e.g.,   │
                       │  concat/average + FFN update)   │
                       └────────────────┬────────────────┘
                                        │
                             ┌──────────▼──────────┐
                             │    Transformer      │
                             │    Decoder Layers   │
                             └──────────┬──────────┘
                                        │
                             ┌──────────▼────────────┐
                             │  3D Box Predictions   │
                             │ (class + box params)  │
                             └───────────────────────┘


| Stage                                 | Description                                    | Shape                                                                                    |
| ------------------------------------- | ---------------------------------------------- | ---------------------------------------------------------------------------------------- |
| **1. Input RGB Images**               | Multi-view images from `K` cameras             | `[B, K, 3, H, W]`                                                                        |
| **2. CNN Backbone (per camera)**      | Feature extraction (e.g., ResNet)              | `[B, K, C_backbone, H/32, W/32]` (e.g., `C_backbone=2048`)                               |
| **3. FPN / Neck**                     | Multi-scale features, unified channel size     | `[B, K, L, C, H_l, W_l]`  <br> (e.g., `L=4`, `C=256`)                                    |
| **4. 2D Feature Maps from Cameras**   | Re-organized for next stage                    | `[B, K, L, C, H_l, W_l]`                                                                 |
| **5. Learned Object Queries**         | Includes embeddings and 3D reference points    | <ul><li>Embeddings: `[B, N, D]` (e.g., `D=256`)</li><li>3D points: `[B, N, 3]`</li></ul> |
| **6. Project 3D Ref Points → 2D**     | Via camera extrinsics/intrinsics               | `[B, K, N, 2]`                                                                           |
| **7. Sample Features (grid\_sample)** | Bilinear sampling at projected 2D points       | `[B, K, N, C]`                                                                           |
| **8. Fuse Features**                  | Across `K` cameras (e.g., avg or concat + MLP) | `[B, N, C]` (or `[B, N, K*C] → MLP → [B, N, C]`)                                         |
| **9. Transformer Decoder**            | Cross-attn + FFN updates per object            | `[B, N, D]` (e.g., `[B, 900, 256]`)                                                      |
| **10. Final Heads**                   | Class + box regression                         | `[B, N, num_classes + 7]`                                                                |


### 2.1 Backbone + FPN

                 +------------------+
                 |   Multi-view     |   (e.g., 6 cameras)
                 |   RGB Images     |
                 +--------+---------+
                          |
               +----------v-----------+
               |    CNN Backbone      |  (e.g., ResNet-101)
               +----------+-----------+
                          |
               +----------v-----------+
               |   Feature Pyramid    |
               |     Network (FPN)    |  --> Outputs: P2, P3, P4, P5
               +----------+-----------+
                          |
             +------------+------------+
             |                         |
     +-------v------+         +--------v-------+     For each 3D query:
     |  High-res     |  ...   |  Low-res        |     - Project to 2D (via camera extrinsics/intrinsics)
     |  Feature Map  |        |  Feature Map    |     - Use `grid_sample` to extract features
     +-------+------+         +--------+--------+
             |                          |
         (sampled using             (sampled using
         projected 2D pts)          projected 2D pts)
             |                          |
         +---v--------------------------v---+
         |     Sampled Image Features       |
         +---------------+------------------+
                         |
               +---------v----------+
               |   DETR Decoder     |
               | (w/ 3D queries)    |
               +---------+----------+
                         |
             +-----------v-----------+
             |     3D Boxes Output    |
             +------------------------+


In [ ]:
class SimpleBackboneFPN(nn.Module):
    """Dummy backbone + FPN: returns multi-scale features for K images"""
    def __init__(self, in_channels=3, feat_channels=256):
        super().__init__()
        # For simplicity, just dummy convs for multi-level features
        self.levels = nn.ModuleList([
            nn.Conv2d(in_channels, feat_channels, 3, stride=2, padding=1),  # Level 1
            nn.Conv2d(feat_channels, feat_channels, 3, stride=2, padding=1), # Level 2
            nn.Conv2d(feat_channels, feat_channels, 3, stride=2, padding=1), # Level 3
            nn.Conv2d(feat_channels, feat_channels, 3, stride=2, padding=1), # Level 4
        ])
    
    def forward(self, images):
        # images: [B, K, C, H, W]
        B, K, C, H, W = images.shape
        features_per_level = []
        for conv in self.levels:
            # Process each camera image independently
            feat_list = []
            for k in range(K):
                x = images[:, k]  # [B, C, H, W]
                x = conv(x)       # [B, feat_channels, H_out, W_out]
                feat_list.append(x)
            features_per_level.append(torch.stack(feat_list, dim=1))  # [B, K, C, H_out, W_out]
            H, W = x.shape[-2], x.shape[-1]
        # features_per_level: list of 4 tensors, each [B, K, C, H_lvl, W_lvl]
        return features_per_level

### 2.2 Reference Point Decoder

In [ ]:
class ReferencePointDecoder(nn.Module):
    """Φ_ref neural network to decode 3D reference point from query"""
    def __init__(self, embed_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 3),  # output (x,y,z) in 3D
            nn.Sigmoid()  # normalized coordinates in [0,1], can be rescaled
        )
    
    def forward(self, queries):
        # queries: [B, M, C]
        return self.fc(queries)  # [B, M, 3]

In [ ]:
def project_to_image(ref_points_h, cam_matrices):
    """
    Project 3D homogeneous points to 2D image coords
    ref_points_h: [B, M, 4] homogeneous coordinates (x,y,z,1)
    cam_matrices: [B, K, 3, 4] camera projection matrices
    returns: normalized image coords [B, M, K, 2] in [-1,1] for grid_sample
    """
    B, M, _ = ref_points_h.shape
    K = cam_matrices.shape[1]
    
    # Expand for each camera
    ref_points_exp = ref_points_h.unsqueeze(2).repeat(1,1,K,1)  # [B, M, K, 4]
    cam_matrices = cam_matrices.unsqueeze(1).repeat(1, M, 1, 1, 1)  # [B, M, K, 3, 4]
    
    # Matrix multiply
    proj = torch.matmul(cam_matrices, ref_points_exp.unsqueeze(-1))  # [B, M, K, 3, 1]
    proj = proj.squeeze(-1)  # [B, M, K, 3]
    
    # Normalize by last coordinate to get 2D pixel coords
    xy = proj[..., :2] / (proj[..., 2:3] + 1e-6)  # [B, M, K, 2]
    
    # Here you must normalize xy by image size (W,H) and map to [-1,1]
    # Assuming known image sizes W_img, H_img (hardcoded or passed in)
    W_img, H_img = 1600, 900
    xy_norm = xy.clone()
    xy_norm[..., 0] = (xy[..., 0] / W_img) * 2 - 1  # x: 0~W -> -1~1
    xy_norm[..., 1] = (xy[..., 1] / H_img) * 2 - 1  # y: 0~H -> -1~1
    return xy_norm  # [B, M, K, 2]

In [ ]:
def bilinear_sample(features, coords):
    """
    Sample features via bilinear interpolation from multi-camera multi-level features.
    features: [B, K, C, H, W]
    coords: [B, M, K, 2] normalized coords in [-1,1]
    returns sampled features: [B, M, C]
    """
    B, K, C, H, W = features.shape
    M = coords.shape[1]
    sampled_feats = []
    for k in range(K):
        feat = features[:, k]  # [B, C, H, W]
        grid = coords[:, :, k].unsqueeze(1)  # [B, M, 1, 2]
        # grid_sample expects [B, C, H, W] input and [B, M, 1, 2] grid
        sampled = F.grid_sample(feat, grid, align_corners=True, mode='bilinear')  # [B, C, M, 1]
        sampled = sampled.squeeze(-1).permute(0, 2, 1)  # [B, M, C]
        sampled_feats.append(sampled)
    # Sum or average over cameras
    sampled_feats = torch.stack(sampled_feats, dim=0).mean(dim=0)  # [B, M, C]
    return sampled_feats


### DETR Layer

In [ ]:
class DETR3DLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, num_cameras=6):
        super().__init__()
        self.ref_point_decoder = ReferencePointDecoder(embed_dim)
        self.self_attn = MultiHeadSelfAttention(embed_dim, num_heads)
        self.num_cameras = num_cameras
        self.eps = 1e-6
    
    def forward(self, queries, features, cam_matrices):
        """
        queries: [B, M, C]
        features: list of 4 feature levels, each [B, K, C, H, W]
        cam_matrices: [B, K, 3, 4]
        """
        B, M, C = queries.shape
        
        # 1. Decode 3D centers from queries
        ref_points = self.ref_point_decoder(queries)  # [B, M, 3]
        
        # 2. Convert to homogeneous coordinates: [x,y,z,1]
        ref_points_h = torch.cat([ref_points, torch.ones(B, M, 1, device=queries.device)], dim=-1)  # [B, M, 4]
        
        # 3. Project to 2D normalized image coordinates per camera & level
        # We'll do for just one level here for simplicity, ideally sample from all 4 levels and fuse
        coords = project_to_image(ref_points_h, cam_matrices)  # [B, M, K, 2]
        
        # 4. Sample features from image features at coords (for one level)
        sampled_feats = bilinear_sample(features[0], coords)  # [B, M, C]
        
        # 5. Heuristic mask to filter points projected outside image (coords outside [-1,1])
        valid_mask = ((coords >= -1.0) & (coords <= 1.0)).all(dim=-1).float()  # [B, M, K]
        valid_mask_sum = valid_mask.sum(dim=-1, keepdim=True) + self.eps  # [B, M, 1]
        # Weighted average
        f_i = (sampled_feats * valid_mask.mean(dim=-1, keepdim=True))  # simplification
        
        # 6. Update queries by adding sampled features
        updated_queries = queries + f_i
        
        # 7. Multi-head self-attention among queries
        # Note nn.MultiheadAttention expects [seq_len, batch, embed_dim]
        q_in = updated_queries.permute(1, 0, 2)  # [M, B, C]
        attn_out = self.self_attn(q_in)
        attn_out = attn_out.permute(1, 0, 2)  # [B, M, C]
        
        # Return updated queries for next layer
        return attn_out

### Full DETR

In [ ]:
class DETR3DModel(nn.Module):
    def __init__(self, num_queries=900, num_classes=10, embed_dim=256, num_heads=8, num_layers=6, num_cameras=6):
        super().__init__()
        self.backbone_fpn = SimpleBackboneFPN()
        self.transformer_layers = nn.ModuleList([
            DETR3DLayer(embed_dim, num_heads, num_cameras) for _ in range(num_layers)
        ])
        self.box_predictor = BoxPredictor(embed_dim, num_classes)
        self.query_embed = nn.Embedding(num_queries, embed_dim)
    
    def forward(self, images, cam_matrices):
        """
        images: [B, K, 3, H, W]
        cam_matrices: [B, K, 3, 4]
        """
        B, K, C, H, W = images.shape
        
        # Backbone + FPN to get multi-scale features
        features = self.backbone_fpn(images)  # list of 4 tensors, each [B, K, C, H_lvl, W_lvl]
        
        # Init object queries (expand batch)
        queries = self.query_embed.weight.unsqueeze(0).repeat(B, 1, 1)  # [B, num_queries, embed_dim]
        
        # Pass through L transformer decoder layers
        for layer in self.transformer_layers:
            queries = layer(queries, features, cam_matrices)
        
        # Predict boxes and classes
        bboxes, class_logits = self.box_predictor(queries)
        
        return bboxes, class_logits

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F



class BoxPredictor(nn.Module):
    """Φ_reg and Φ_cls networks for bbox and class prediction"""
    def __init__(self, embed_dim, num_classes):
        super().__init__()
        self.regressor = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 9),  # bbox params: pos(3), size(3), rot(1), velocity(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, num_classes),
        )
    
    def forward(self, queries):
        bbox = self.regressor(queries)   # [B, M, 9]
        cls_logits = self.classifier(queries)  # [B, M, num_classes]
        return bbox, cls_logits



class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim, num_heads)
    
    def forward(self, x):
        # x: [M, B, C] (queries first dimension for nn.MultiheadAttention)
        attn_output, _ = self.mha(x, x, x)
        return attn_output







